# Automatic CLEAN Mask Demo

[Colab Link](https://colab.research.google.com/github/casangi/astroviper/blob/main/docs/processing_functions_tutorials/image_analysis/make_auto_mask.ipynb)

`make_auto_mask` makes the CLEAN mask from the residual image, with the algorithm of CASA's `usemask='auto-multithresh'` (Kepley et al. 2020, PASP 132, 024505). `image_cube_single_field` and `image_continuum_single_field` run it after every residual update when they get `auto_mask_params`.

This notebook calls the function directly on a simulated residual, to show how the mask follows the residual through the imaging cycles.

## Install AstroVIPER

Installs AstroVIPER only where it is missing.

In [ ]:
import os
from importlib.metadata import PackageNotFoundError, version

try:
    print("Using astroviper version", version("astroviper"))
except PackageNotFoundError:
    os.system("pip install --upgrade astroviper[all]")

## API

In [ ]:
from astroviper.processing_functions.image_analysis import make_auto_mask

make_auto_mask?

## What every update does

For every plane:

1. The **sidelobe level** is `sidelobe_threshold` times the largest PSF sidelobe times the peak residual.
2. The mask starts from the pixels above the larger of the sidelobe level and `noise_threshold` times the noise. Regions smaller than `min_beam_fraction` clean beams are removed. What is left is smoothed with the clean beam and cut at `cut_threshold` of its peak.
3. A mask of a preceding residual update also grows through the pixels above the larger of the sidelobe level and `low_noise_threshold` times the noise.
4. With `negative_threshold` above 0, the mask also takes the negative features.

The noise is 1.4826 times the median absolute deviation, and every threshold is relative to the median. A mask never loses pixels. As the peak residual falls, the sidelobe level falls with it, so fainter emission enters the mask.

## Simulate a residual

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from scipy.ndimage import gaussian_filter
from scipy.signal import fftconvolve

A toy PSF: a Gaussian main lobe and a ripple under a wide envelope, with the known largest sidelobe. The sky has a bright compact source, a fainter compact source and a faint extended source. `residual` gives the image when the model has taken up all but the given share of each source.

In [ ]:
N_PIXELS = 256
CELL = np.deg2rad(0.1 / 3600.0)
BEAM_PIXELS = 7.0
BEAM = np.array([BEAM_PIXELS, BEAM_PIXELS, 0.0]) * [CELL, CELL, 1.0]
NOISE = 0.01

offsets = np.arange(N_PIXELS) - N_PIXELS // 2
x, y = np.meshgrid(offsets, offsets, indexing="ij")
radius_squared = x**2 + y**2
sigma = BEAM_PIXELS / np.sqrt(8.0 * np.log(2.0))
main_lobe = np.exp(-radius_squared / (2.0 * sigma**2))
ripple = np.cos(2.0 * np.pi * (x + 0.6 * y) / (3.0 * BEAM_PIXELS))
envelope = np.exp(-radius_squared / (2.0 * (12.0 * sigma) ** 2))
psf = main_lobe + 0.12 * ripple * envelope * (1.0 - main_lobe)
sidelobe = np.abs(psf[radius_squared > (2.0 * BEAM_PIXELS) ** 2]).max()


def compact_source(flux, offset_l, offset_m):
    sky = np.zeros((N_PIXELS, N_PIXELS))
    sky[N_PIXELS // 2 + offset_l, N_PIXELS // 2 + offset_m] = flux
    return sky


bright = compact_source(10.0, 0, 0)
faint = compact_source(0.8, -55, 45)
extended = 0.25 * np.exp(-((x - 50) ** 2 + (y + 40) ** 2) / (2.0 * 14.0**2))

rng = np.random.default_rng(7)
noise = gaussian_filter(rng.normal(size=(N_PIXELS, N_PIXELS)), sigma)
noise *= NOISE / noise.std()


def residual(bright_left, faint_left, extended_left):
    sky = bright_left * bright + faint_left * faint
    return fftconvolve(sky, psf, mode="same") + extended_left * extended + noise


f"largest sidelobe {sidelobe:.3f}, sidelobe level of the dirty image {3 * sidelobe * 10:.2f}"

`make_auto_mask` works on an image dataset with the dimensions `(time, frequency, polarization, l, m)`. The data group holds the residual and the clean beam and largest sidelobe of every plane; the imagers fill these in. It may hold a `primary_beam`, and the mask of the preceding update.

In [ ]:
def image_dataset(plane):
    planes = ("time", "frequency", "polarization")
    return xr.Dataset(
        {
            "SKY_RESIDUAL": (
                (*planes, "l", "m"),
                plane[np.newaxis, np.newaxis, np.newaxis],
            ),
            "BEAM": ((*planes, "beam_param"), BEAM[np.newaxis, np.newaxis, np.newaxis]),
            "SIDELOBE": (planes, np.full((1, 1, 1), sidelobe)),
        },
        coords={"l": offsets * CELL, "m": offsets * CELL},
        attrs={
            "data_groups": {
                "residual": {
                    "sky": "SKY_RESIDUAL",
                    "beam_fit_params_point_spread_function": "BEAM",
                    "max_sidelobe_point_spread_function": "SIDELOBE",
                }
            }
        },
    )

## Follow the mask through three residual updates

The first update sees the dirty image: the sidelobes of the bright source hide the rest, and only that source passes. In the second the bright source is mostly in the model, so the fainter compact source passes. In the third that source is in the model too, and the sidelobe level no longer hides the extended source.

The mask stays in the image from one update to the next, and `planes_stopped` goes with it.

In [ ]:
shares_left = [(1.0, 1.0, 1.0), (0.05, 1.0, 1.0), (0.01, 0.1, 1.0)]
residuals = [residual(*shares) for shares in shares_left]

image = image_dataset(residuals[0])
planes_stopped = None
masks = []
for update, plane in enumerate(residuals):
    image["SKY_RESIDUAL"].values[0, 0, 0] = plane
    planes_stopped = make_auto_mask(
        image, auto_mask_params={}, planes_stopped=planes_stopped
    )
    masks.append(image.MASK.values[0, 0, 0].copy())
    print(
        f"update {update + 1}: peak {np.abs(plane).max():.2f}, "
        f"sidelobe level {3.0 * sidelobe * np.abs(plane).max():.3f}, "
        f"mask {masks[-1].sum()} pixels"
    )

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(13, 4.2), constrained_layout=True)
for update, (axis, plane, shown) in enumerate(zip(axes, residuals, masks, strict=True)):
    scale = np.abs(plane).max()
    axis.imshow(
        plane.T, origin="lower", cmap="gray_r", vmin=-0.2 * scale, vmax=0.6 * scale
    )
    axis.contour(shown.T, levels=[0.5], colors="tab:red", linewidths=1.0)
    axis.set_title(f"update {update + 1}: peak {scale:.2f} Jy/beam")
    axis.set_xticks([])
    axis.set_yticks([])

## An image of noise

Where the noise sets the threshold and the mask stays empty, the plane stops: `make_auto_mask` returns it in `planes_stopped` and leaves it alone from then on. The imagers do the same, and such a plane ends with stop code 7.

In [ ]:
image = image_dataset(noise)
planes_stopped = make_auto_mask(image, auto_mask_params={})
int(image.MASK.values.sum()), bool(planes_stopped[0, 0, 0])

## In the imagers

`auto_mask_params` takes the same keys in `image_cube_single_field` and `image_continuum_single_field`. An empty dict gives the `tclean` defaults, and `None`, the default, makes no automatic mask. It is refused together with `clean_mask`.

```python
image_cube_single_field(
    ...,
    auto_mask_params={"sidelobe_threshold": 2.0, "noise_threshold": 4.25},
)
```

| `auto_mask_params` key | `tclean` parameter | Default |
| --- | --- | --- |
| `sidelobe_threshold` | `sidelobethreshold` | 3.0 |
| `noise_threshold` | `noisethreshold` | 5.0 |
| `low_noise_threshold` | `lownoisethreshold` | 1.5 |
| `negative_threshold` | `negativethreshold` | 0.0 |
| `smooth_factor` | `smoothfactor` | 1.0 |
| `min_beam_fraction` | `minbeamfrac` | 0.3 |
| `cut_threshold` | `cutthreshold` | 0.01 |
| `grow_iterations` | `growiterations` | 75 |
| `grow_prune` | `dogrowprune` | `True` |
| `min_percent_change` | `minpercentchange` | -1.0 |
| `fast_noise` | `fastnoise` | `True` |

Differences from CASA: the clean beam and the largest sidelobe are those of each plane, not of the whole cube, and a plane stops updating on its own. `threshold_per_cycle` is per plane too, so a faint channel cleans deeper before its next mask update than in CASA. The mask written includes the update after the last model update, which CASA computes but does not keep.